# Notebook 08 — Daily Sentiment Aggregation

This notebook does exactly one thing: it takes the per-document FinBERT sentiment
file produced in Step 3 (`layer2_sentiment.parquet`) and turns it into a daily-grain
table aligned to the NYSE trading calendar. The output, `daily_sentiment.parquet`,
is consumed directly by Notebook 09, which builds the lag/rolling/momentum/EMA
features and the BUY/SELL/HOLD targets.

**No feature engineering, no targets, no splits in this notebook.** Those belong in
Notebook 09. Keeping the two stages separate makes ablation studies (e.g. *does
engagement weighting help?*) trivial to run later without re-doing the daily roll-up.

The full design rationale lives in the Step 4 Implementation Specification PDF;
the cells below cite the relevant sections (§3.x) where useful.

**Pipeline position.**

`layer2_sentiment.parquet` (Step 3, ~85,647 rows) → **this notebook** → `daily_sentiment.parquet` (long format, ~5,000 rows).


## Setup

Imports, paths, and project constants. The seed is fixed to keep the small
amount of randomness in this notebook (the optional sample sanity check)
reproducible.

In [1]:
# Standard scientific Python stack
import os
import json
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# pandas-market-calendars is the cleanest way to get the NYSE trading calendar.
# Falling back to a manual hardcoded list would be brittle around half-day holidays.
try:
    import pandas_market_calendars as mcal
except ImportError:
    !pip install pandas_market_calendars -q
    import pandas_market_calendars as mcal

print("pandas:", pd.__version__)
print("numpy: ", np.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.0/131.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.3/213.3 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 2.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ibis-framework 9.5.0 requires toolz<1,>=0.11, but you have toolz 1.1.0 which is incompatible.
pandas: 2.3.3
numpy:  2.0.2


In [2]:
# --- CONFIG --------------------------------------------------------------
# Kaggle paths. Adjust the dataset folder names to match the input I attach
# to this notebook on Kaggle. Everything else flows from these.

INPUT_DIR  = "/kaggle/input"
OUTPUT_DIR = "/kaggle/working"

# Step 3 output (per-document FinBERT sentiment). Required input.
LAYER2_SENTIMENT_PATH = f"{INPUT_DIR}/pfe-step3/layer2_sentiment.parquet"

# Layer 1 raw files. Engagement metrics were intentionally dropped between
# Layer 1 and Layer 2 (see brainstorm doc), so I need to re-join here to
# reconstruct them per source.
RAW_TWITTER_GENERAL_PATH = f"{INPUT_DIR}/pfe-step1/raw_twitter_general.parquet"
RAW_TWITTER_MUSK_PATH    = f"{INPUT_DIR}/pfe-step1/raw_twitter_musk.parquet"
RAW_REDDIT_PATH          = f"{INPUT_DIR}/pfe-step1/raw_reddit.parquet"
RAW_NEWS_PATH            = f"{INPUT_DIR}/pfe-step1/raw_news.parquet"

# Output of this notebook.
OUTPUT_PATH = f"{OUTPUT_DIR}/daily_sentiment.parquet"

# Project-wide constants.
SEED = 42
np.random.seed(SEED)

# Date window the project operates on. The buffer days extend the NYSE
# calendar lookup either side, so documents at the edge of the window
# don't silently fail to find a trading day.
WINDOW_START = pd.Timestamp("2020-01-01", tz="UTC")
WINDOW_END   = pd.Timestamp("2023-12-31 23:59:59", tz="UTC")
CAL_BUFFER_START = "2019-12-15"
CAL_BUFFER_END   = "2024-01-15"

# Source labels — single source of truth for downstream code.
SOURCES = ["news", "reddit", "twitter_general", "twitter_musk"]
ALL_SLOT = "all"
ALL_SOURCE_SLOTS = SOURCES + [ALL_SLOT]

print("Config loaded.")

Config loaded.


## 1. Load Step 3 output

`layer2_sentiment.parquet` carries the FinBERT predictions plus everything the
preprocessing pipeline added (`tokens_*`, `entities`, `has_tsla_entity`,
`is_too_short`). I only need the columns relevant to aggregation; everything else
is dropped here to keep memory under control.

In [3]:
df_sent = pd.read_parquet(LAYER2_SENTIMENT_PATH)

# Columns I actually need for daily aggregation. The rest stay in the file
# on disk but don't get loaded into the working dataframe.
needed = [
    "doc_id",
    "published_at",
    "source",
    "primary_label",
    "score_pos",
    "score_neg",
    "score_neu",
    "has_tsla_entity",
    "is_too_short",
]
missing = [c for c in needed if c not in df_sent.columns]
assert not missing, f"Missing required columns from Step 3 output: {missing}"

df_sent = df_sent[needed].copy()

# Make sure published_at is UTC-aware. If Step 3 stripped the timezone,
# I'll attach it back here — every doc was collected as UTC originally.
if df_sent["published_at"].dt.tz is None:
    df_sent["published_at"] = df_sent["published_at"].dt.tz_localize("UTC")

print(f"Loaded {len(df_sent):,} rows from Step 3.")
print()
print("Source distribution:")
print(df_sent["source"].value_counts())
print()
print("Primary label distribution:")
print(df_sent["primary_label"].value_counts(dropna=False))

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/pfe-step3/layer2_sentiment.parquet'

## 2. Filtering + quality report  (spec §5)

Three categories of documents are excluded before aggregation:

1. `has_tsla_entity = False` — documents that don't actually reference Tesla. They
   were retained in Step 2 deliberately for transparency, but they only add noise
   to a TSLA-sentiment aggregate.
2. `is_too_short = True` — texts under 3 tokens after cleaning. FinBERT predictions
   on near-empty strings are unreliable.
3. `NaN` in `primary_label` or in any of the three score columns — Step 3 failed
   to produce a usable label for whatever reason.

I record per-source counts before and after filtering so the totals can go into
the thesis data-quality section.

In [ ]:
# Snapshot the pre-filter counts.
before = df_sent.groupby("source").size().rename("before")

# Drop in three explicit passes so the quality report can show what each pass costs.
mask_no_tsla   = df_sent["has_tsla_entity"] == False  # noqa: E712 — explicit bool compare
mask_too_short = df_sent["is_too_short"] == True      # noqa: E712
mask_nan_sent  = (
    df_sent["primary_label"].isna()
    | df_sent["score_pos"].isna()
    | df_sent["score_neg"].isna()
    | df_sent["score_neu"].isna()
)

drops = pd.DataFrame({
    "before":           before,
    "drop_no_tsla":     df_sent[mask_no_tsla].groupby("source").size(),
    "drop_too_short":   df_sent[mask_too_short].groupby("source").size(),
    "drop_nan_sent":    df_sent[mask_nan_sent].groupby("source").size(),
}).fillna(0).astype(int)

# Apply all three filters in one pass.
keep = ~(mask_no_tsla | mask_too_short | mask_nan_sent)
df_sent = df_sent[keep].copy()

drops["after"]   = df_sent.groupby("source").size()
drops = drops.fillna(0).astype(int)
drops["pct_kept"] = (drops["after"] / drops["before"] * 100).round(2)

print("Per-source filtering report:")
print(drops)
print()
print(f"Total rows after filtering: {len(df_sent):,}")

## 3. Re-attach engagement metrics from Layer 1

Engagement metrics (likes, retweets, upvotes, follower counts, news source name)
were intentionally excluded from Layer 2 to keep the NLP-pipeline input clean.
They re-enter the project here because they're the inputs to the engagement
weights computed in §5.

Strategy: load each Layer 1 raw parquet, keep only the engagement columns plus
the deduplication key (`url` for news, the per-source primary ID otherwise),
and left-join into `df_sent` on a shared key. The Step 3 file already carries
`doc_id` and the original `url`, so I match on whichever is most reliable per
source.

> **Note on the doc_id collision detected in Notebook 06.** A handful of tweets
> with missing `tweet_id` shared the URL `https://twitter.com/i/web/status/None`,
> producing collapsed `doc_id`s. After Step 3's dedupe-on-doc_id-keep-first this
> issue is contained, but I'm logging the pre/post-merge row counts below to
> catch any regression.

In [ ]:
def _load_raw(path, keep_cols, source_label):
    """Helper: load a Layer 1 raw parquet, keep only the columns I need."""
    df = pd.read_parquet(path)
    available = [c for c in keep_cols if c in df.columns]
    df = df[available].copy()
    df["_source_check"] = source_label
    return df

# --- Twitter (general + Musk) --------------------------------------------
# Engagement signal: likes, retweets, replies, author_followers.
# Match key: url (most stable across the pipeline).
tw_cols = ["url", "likes", "retweets", "replies", "author_followers"]
df_tw_gen  = _load_raw(RAW_TWITTER_GENERAL_PATH, tw_cols, "twitter_general")
df_tw_musk = _load_raw(RAW_TWITTER_MUSK_PATH,    tw_cols, "twitter_musk")

# --- Reddit ---------------------------------------------------------------
# Engagement signal: upvotes, upvote_ratio, num_comments.
red_cols = ["url", "upvotes", "upvote_ratio", "num_comments"]
df_reddit = _load_raw(RAW_REDDIT_PATH, red_cols, "reddit")

# --- News -----------------------------------------------------------------
# No raw engagement on news; what I do have is the outlet name, which I'll
# convert to a credibility score in §5.
news_cols = ["url", "source_name"]
df_news = _load_raw(RAW_NEWS_PATH, news_cols, "news")

# Concatenate everything into one engagement table keyed by (url, source).
# I keep `_source_check` so I can verify the join is matching the right rows
# for each source slot (and I don't accidentally pull a Reddit upvote count
# onto a Twitter doc that happens to share a URL).
df_eng = pd.concat([df_tw_gen, df_tw_musk, df_reddit, df_news],
                   ignore_index=True)

# Reload Step 3 file's url column for matching. It's already in df_sent for
# rows where it survived Step 2; if any row has a NaN url I'll fill it with
# doc_id so the merge doesn't silently drop it.
if "url" not in df_sent.columns:
    # Step 3 may have dropped url to save space — reload it from Layer 2 raw.
    df_l2 = pd.read_parquet(LAYER2_SENTIMENT_PATH, columns=["doc_id", "url"])
    df_sent = df_sent.merge(df_l2, on="doc_id", how="left")

# Merge engagement onto the sentiment df. The (url, source) key prevents
# the cross-source URL collision case mentioned above.
n_before = len(df_sent)
df = df_sent.merge(
    df_eng,
    left_on=["url", "source"],
    right_on=["url", "_source_check"],
    how="left",
)
df = df.drop(columns=["_source_check"])
n_after = len(df)
assert n_before == n_after, (
    f"Engagement merge changed row count: {n_before} -> {n_after}. "
    "This means the (url, source) key isn't unique on the right side."
)

# Coverage check — what fraction of rows actually got engagement data.
print("Engagement merge coverage by source:")
for s in SOURCES:
    sub = df[df["source"] == s]
    if s == "news":
        cov = sub["source_name"].notna().mean()
        print(f"  {s:18s} {len(sub):7,} rows, {cov:6.1%} have source_name")
    elif s.startswith("twitter"):
        cov = sub["likes"].notna().mean()
        print(f"  {s:18s} {len(sub):7,} rows, {cov:6.1%} have likes")
    elif s == "reddit":
        cov = sub["upvotes"].notna().mean()
        print(f"  {s:18s} {len(sub):7,} rows, {cov:6.1%} have upvotes")

## 4. Trading-day alignment  (spec §3.2)

The rule is: take each document's `published_at` UTC timestamp, convert to
America/New_York, and assign to a trading day. If local time is ≥ 16:00 EST,
the document is rolled to the next trading day (its sentiment can't have
caused today's move because the market was already closed). Weekends and
NYSE holidays roll forward until they hit a real trading day.

This single rule eliminates the most common time-alignment bug in
sentiment-driven trading projects.

In [ ]:
# Build the NYSE trading-day index once. The buffer either side of the
# project window catches edge cases (e.g. a Sunday-night document whose
# rolled trading day is the following Monday).
nyse = mcal.get_calendar("NYSE")
schedule = nyse.schedule(start_date=CAL_BUFFER_START, end_date=CAL_BUFFER_END)
trading_days = pd.DatetimeIndex(schedule.index.normalize())

# Convert to a sorted numpy array of date-only values for fast searchsorted.
trading_days_arr = trading_days.to_numpy()
print(f"NYSE trading-day index spans {trading_days_arr[0].astype('datetime64[D]')} "
      f"to {trading_days_arr[-1].astype('datetime64[D]')} "
      f"({len(trading_days_arr)} days).")

In [ ]:
def assign_trading_day_vec(ts_utc):
    """Vectorised trading-day assignment.

    For each timestamp:
      1. Convert UTC -> America/New_York.
      2. Take the calendar date in NY.
      3. If local hour >= 16, advance one calendar day.
      4. Snap to the next NYSE trading day >= that date.

    Returns a tz-naive Timestamp index of trading days (one per input row).
    """
    ts_ny = ts_utc.dt.tz_convert("America/New_York")

    # Step (2) + (3): candidate calendar date, advanced if past close.
    cal_date = ts_ny.dt.normalize().dt.tz_localize(None)
    after_close = ts_ny.dt.hour >= 16
    cal_date = cal_date + pd.to_timedelta(after_close.astype(int), unit="D")

    # Step (4): snap forward to next trading day. searchsorted with side='left'
    # gives the index of the first trading day >= cal_date.
    idx = np.searchsorted(trading_days_arr, cal_date.to_numpy(), side="left")
    # Guard against running off the end of the calendar (shouldn't happen with
    # the buffer, but defensive is cheap).
    idx = np.clip(idx, 0, len(trading_days_arr) - 1)
    return pd.to_datetime(trading_days_arr[idx])

df["trading_day"] = assign_trading_day_vec(df["published_at"])

# Restrict to the project window. Documents outside [2020, 2023] either rolled
# in from late 2019 or dangling 2024-01 timestamps; either way they're outside
# the analysis window.
in_window = (
    (df["trading_day"] >= pd.Timestamp("2020-01-01"))
    & (df["trading_day"] <= pd.Timestamp("2023-12-31"))
)
df = df[in_window].copy()

print(f"Rows after trading-day assignment + window filter: {len(df):,}")
print()
print("Documents per year (post-roll):")
print(df["trading_day"].dt.year.value_counts().sort_index())

In [ ]:
# Quick sanity check: what fraction of documents got rolled to a *later*
# day than their UTC calendar date? This should be non-trivial — anything
# at or after 16:00 EST plus all weekend/holiday traffic.
naive_day = df["published_at"].dt.tz_convert("America/New_York").dt.normalize().dt.tz_localize(None)
rolled = (df["trading_day"] != naive_day).mean()
print(f"Fraction of documents rolled to a later trading day: {rolled:.1%}")

## 5. Engagement weights  (spec §3.3)

Each document gets a positive weight `w_doc` that's used inside the daily
weighted aggregations in §6. The weight is source-specific: Twitter uses the
log-scaled engagement and follower count, Reddit uses upvotes scaled by
upvote ratio, news uses a small credibility lookup table.

All weights are floored at 1.0 so a document with zero recorded engagement
still contributes (just at the lowest possible weight). The `log1p` is
*essential* — without it one viral tweet with 500K likes would dominate every
aggregate it appears in.

> **Soft assumption flagged.** The exact coefficients in the Twitter formula
> (2× retweets, 0.5× replies) and the news credibility lookup table are
> defensible defaults but not empirically tuned. The Step 4 spec marks both
> as items to confirm with the supervisor.

In [ ]:
# News credibility lookup. Uppercase the lookup keys for case-insensitive
# matching since outlet names appear with inconsistent casing in GDELT.
NEWS_CREDIBILITY = {
    "REUTERS":          3.0,
    "BLOOMBERG":        3.0,
    "WALL STREET JOURNAL": 3.0,
    "WSJ":              3.0,
    "FINANCIAL TIMES":  3.0,
    "FT":               3.0,
    "THE NEW YORK TIMES": 2.5,
    "NYT":              2.5,
    "CNBC":             2.0,
    "FORBES":           2.0,
    "THE GUARDIAN":     2.0,
    "BUSINESS INSIDER": 1.5,
    "MARKETWATCH":      1.5,
    "SEEKING ALPHA":    1.5,
    "THE MOTLEY FOOL":  1.5,
    "YAHOO FINANCE":    1.5,
    # Default for anything not listed: 1.0 — applied below.
}


def weight_twitter(row):
    """Engagement weight for a single Twitter document."""
    likes     = row.get("likes", 0) or 0
    retweets  = row.get("retweets", 0) or 0
    replies   = row.get("replies", 0) or 0
    followers = row.get("author_followers", 0) or 0
    eng = np.log1p(likes + 2.0 * retweets + 0.5 * replies)
    foll = np.log1p(followers)
    return max(eng * foll, 1.0)


def weight_reddit(row):
    """Engagement weight for a single Reddit document."""
    upvotes = row.get("upvotes", 0) or 0
    ratio   = row.get("upvote_ratio", 0.5)
    if pd.isna(ratio):
        ratio = 0.5
    comments = row.get("num_comments", 0) or 0
    main = np.log1p(upvotes) * max(ratio, 0.5)
    sec  = 0.5 * np.log1p(comments)
    return max(main + sec, 1.0)


def weight_news(row):
    """Credibility-based weight for a single news document."""
    name = row.get("source_name")
    if pd.isna(name):
        return 1.0
    return NEWS_CREDIBILITY.get(str(name).upper(), 1.0)


# Vectorised computation per source — apply() is fine here, the dataset is
# ~85K rows and this only runs once.
df["w"] = 1.0  # default

mask_tw  = df["source"].isin(["twitter_general", "twitter_musk"])
mask_red = df["source"] == "reddit"
mask_news = df["source"] == "news"

df.loc[mask_tw,  "w"] = df[mask_tw].apply(weight_twitter, axis=1)
df.loc[mask_red, "w"] = df[mask_red].apply(weight_reddit, axis=1)
df.loc[mask_news,"w"] = df[mask_news].apply(weight_news, axis=1)

# Sanity: every weight must be finite and >= 1.
assert df["w"].notna().all() and (df["w"] >= 1.0).all(), "Bad engagement weights."

print("Engagement weight distribution per source:")
print(df.groupby("source")["w"].describe().round(2))

## 6. Per-(trading_day, source) aggregation  (spec §3.4)

Six statistics per cell:

| column | definition |
|---|---|
| `vol` | document count |
| `sent_mean_w` | engagement-weighted mean of $s_i = P(\text{pos})_i - P(\text{neg})_i$ |
| `sent_net_w` | engagement-weighted net ratio: $\frac{\sum w_i \cdot \mathbb{1}[\text{pos}] - \sum w_i \cdot \mathbb{1}[\text{neg}]}{\sum w_i}$ |
| `sent_disp_w` | engagement-weighted standard deviation of $s_i$ |
| `n_pos`, `n_neu`, `n_neg` | raw (unweighted) counts of FinBERT-predicted classes |

The class counts are unweighted by design — they're a lossless record of how
many documents of each polarity hit each (day, source) cell, before any
engagement weighting.

In [ ]:
# Step 1: derive the continuous score s = P(pos) - P(neg).
df["s"] = df["score_pos"] - df["score_neg"]

# Step 2: indicator columns for the weighted-net-ratio and the raw counts.
df["is_pos"] = (df["primary_label"] == "positive").astype(int)
df["is_neu"] = (df["primary_label"] == "neutral").astype(int)
df["is_neg"] = (df["primary_label"] == "negative").astype(int)

# Step 3: products that turn weighted aggregates into simple sums.
df["w_s"]     = df["w"] * df["s"]
df["w_pos"]   = df["w"] * df["is_pos"]
df["w_neg"]   = df["w"] * df["is_neg"]


def aggregate_by(df_in, group_cols):
    """Compute the six daily statistics over a given groupby key.

    Used twice: once for (trading_day, source) and once for trading_day alone
    (the global 'all' slot). Returns a tidy dataframe with one row per group
    and the column schema defined above.
    """
    g = df_in.groupby(group_cols, observed=True)

    sum_w     = g["w"].sum()
    sum_w_s   = g["w_s"].sum()
    sum_w_pos = g["w_pos"].sum()
    sum_w_neg = g["w_neg"].sum()
    vol       = g.size().rename("vol")
    n_pos     = g["is_pos"].sum().rename("n_pos")
    n_neu     = g["is_neu"].sum().rename("n_neu")
    n_neg     = g["is_neg"].sum().rename("n_neg")

    sent_mean_w = (sum_w_s / sum_w).rename("sent_mean_w")
    sent_net_w  = ((sum_w_pos - sum_w_neg) / sum_w).rename("sent_net_w")

    # Weighted std, computed via the identity:
    #   var_w(x) = E_w[x^2] - E_w[x]^2
    #            = sum(w * x^2) / sum(w) - (sum(w * x) / sum(w))^2
    # This avoids per-group apply() and stays vectorised. For groups with
    # vol < 2 the variance is undefined; I set those to NaN explicitly.
    df_in_local = df_in.assign(_w_s2=df_in["w"] * df_in["s"] ** 2)
    sum_w_s2 = df_in_local.groupby(group_cols, observed=True)["_w_s2"].sum()
    var_w = sum_w_s2 / sum_w - sent_mean_w ** 2
    var_w = var_w.clip(lower=0)  # guard against tiny negative values from float error
    sent_disp_w = np.sqrt(var_w).rename("sent_disp_w")
    sent_disp_w[vol < 2] = np.nan

    out = pd.concat(
        [vol, sent_mean_w, sent_net_w, sent_disp_w, n_pos, n_neu, n_neg],
        axis=1,
    ).reset_index()
    return out


# Per-source aggregation.
agg_per_source = aggregate_by(df, ["trading_day", "source"])
print(f"Per-source aggregation: {len(agg_per_source):,} (day, source) cells.")
agg_per_source.head()

## 7. Global 'all' aggregate  (spec §3.5)

The 'all' slot is computed from the document-level data directly — *not* as
a weighted average of the per-source slots. This is important: the engagement
weights are consistent at the document level, but combining per-source means
would require re-deriving compatible weights between sources, which is fragile.

Computing from documents up means the same code path produces both the
per-source and the global aggregates with the same statistical guarantees.

In [ ]:
# Same six statistics, grouping by trading_day alone.
agg_all = aggregate_by(df, ["trading_day"])
agg_all["source"] = ALL_SLOT

# Concatenate per-source + global into one long-format table.
agg = pd.concat([agg_per_source, agg_all], ignore_index=True)
agg = agg[["trading_day", "source",
           "vol", "sent_mean_w", "sent_net_w", "sent_disp_w",
           "n_pos", "n_neu", "n_neg"]]

print(f"Total cells (per-source + all): {len(agg):,}")
print()
print("Cell counts by source slot:")
print(agg["source"].value_counts())

## 8. Build the full daily grid

Some (trading_day, source) cells will be missing from `agg` because no
documents from that source hit that day after filtering. The convention from
the spec (§3.4) is: those cells get `vol = 0` and `NaN` for the four sentiment
columns. This honestly preserves the volume signal (the day really had zero
documents from that source) while telling downstream models that no sentiment
estimate exists.

I build the full cartesian product (NYSE trading days × 5 source slots) and
left-join `agg` onto it.

In [ ]:
# Cartesian product of trading days within the project window × source slots.
project_days = pd.DatetimeIndex(
    [d for d in trading_days_arr
     if pd.Timestamp("2020-01-01") <= pd.Timestamp(d) <= pd.Timestamp("2023-12-31")]
)
print(f"Project trading days: {len(project_days)}")

grid = pd.MultiIndex.from_product(
    [project_days, ALL_SOURCE_SLOTS],
    names=["trading_day", "source"],
).to_frame(index=False)

# Left-join the aggregates onto the full grid.
daily = grid.merge(agg, on=["trading_day", "source"], how="left")

# Apply the zero-volume convention.
fill_zero = ["vol", "n_pos", "n_neu", "n_neg"]
daily[fill_zero] = daily[fill_zero].fillna(0).astype(int)
# sent_mean_w, sent_net_w, sent_disp_w stay NaN where vol == 0.

print(f"Daily grid shape: {daily.shape}")
print(f"  Expected: {len(project_days)} days × {len(ALL_SOURCE_SLOTS)} slots "
      f"= {len(project_days) * len(ALL_SOURCE_SLOTS):,} rows.")
print()
print("Zero-volume cell count by source:")
print(daily.groupby("source").apply(lambda g: (g["vol"] == 0).sum()))

## 9. Save `daily_sentiment.parquet`

Long format on purpose: keeps the file small and makes the lag/rolling code
in Notebook 09 cleaner once it pivots to wide.

In [ ]:
# Type tightening before save — keeps Parquet small and unambiguous.
daily["trading_day"]  = pd.to_datetime(daily["trading_day"]).dt.tz_localize(None)
daily["source"]       = daily["source"].astype("category")
daily["vol"]          = daily["vol"].astype("int32")
daily["n_pos"]        = daily["n_pos"].astype("int32")
daily["n_neu"]        = daily["n_neu"].astype("int32")
daily["n_neg"]        = daily["n_neg"].astype("int32")
daily["sent_mean_w"]  = daily["sent_mean_w"].astype("float32")
daily["sent_net_w"]   = daily["sent_net_w"].astype("float32")
daily["sent_disp_w"]  = daily["sent_disp_w"].astype("float32")

daily = daily.sort_values(["trading_day", "source"]).reset_index(drop=True)

os.makedirs(OUTPUT_DIR, exist_ok=True)
daily.to_parquet(OUTPUT_PATH, index=False)

print(f"Wrote {OUTPUT_PATH}")
print(f"  Rows:    {len(daily):,}")
print(f"  Bytes:   {os.path.getsize(OUTPUT_PATH):,}")
print(f"  Columns: {list(daily.columns)}")

## 10. Quick sanity report

A handful of checks I want to eyeball before handing this off to Notebook 09:

- **Coverage:** the project window has 1,006 NYSE trading days (4 calendar
  years minus weekends and holidays). Every day should appear once per
  source slot.
- **Volume distribution:** per-source histograms of `vol`. If one source
  is consistently zero-volume, something upstream is wrong.
- **Sentiment range:** `sent_mean_w` should sit in roughly [-1, +1] (since
  $s_i \in [-1, +1]$ before weighting).

In [ ]:
# Coverage check.
n_days = daily["trading_day"].nunique()
n_slots = daily["source"].nunique()
expected = n_days * n_slots
print(f"Days × slots = {n_days} × {n_slots} = {expected:,}")
print(f"Actual rows: {len(daily):,}")
assert len(daily) == expected, "Daily grid is incomplete."

print()
print("Volume statistics by source:")
print(daily.groupby("source", observed=True)["vol"].describe().round(1))

In [ ]:
# Sentiment range check.
print("sent_mean_w range by source:")
print(
    daily.dropna(subset=["sent_mean_w"])
         .groupby("source", observed=True)["sent_mean_w"]
         .describe()
         .round(3)
)

In [ ]:
# Quick visual: daily total volume across all sources, week-smoothed.
import matplotlib.pyplot as plt

vol_all = daily[daily["source"] == ALL_SLOT].set_index("trading_day")["vol"]

fig, ax = plt.subplots(figsize=(12, 3.5))
vol_all.rolling(7).mean().plot(ax=ax, lw=1.2)
ax.set_title("Daily document volume (all sources, 7-day rolling mean)")
ax.set_xlabel("")
ax.set_ylabel("documents / day")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

**Done.** `daily_sentiment.parquet` is written to `/kaggle/working/` and ready
to be picked up by Notebook 09 (`09_features_targets_split`). The next stage
will pivot this long table to wide, merge with TSLA OHLC prices, build the
lag/rolling/momentum/EMA features, and produce the BUY/SELL/HOLD targets and
the chronological train/test split.